# Exercise Part 1: Setting Up Your Python Environment on Explorer

In this part of the exercise, you will set up a Python environment on Northeastern's
**Explorer Cluster**, install PyTorch with GPU support, and verify it can see the GPU.

Explorer replaced the older Discovery cluster: newer OS (Rocky Linux 9.3), newer GPUs (H200s,
141 GB memory), and a fresh module system -- so environments built on Discovery may not carry
over cleanly. Build a fresh one here.

**Two ways to manage your environment -- pick one:**

| | Conda | `venv` (recommended) |
|---|---|---|
| Setup | `module load anaconda3` then `conda create` | `module load python/3.10` then `python -m venv` |
| Speed | Slower solver, larger footprint | Faster, uses the stdlib -- nothing extra to load |
| Portability | Conda env is tied to the conda install | A `venv` + `requirements.txt` is trivial to recreate anywhere, incl. off-cluster |
| When to use | You need non-Python deps (e.g. certain CUDA/BLAS builds) that conda packages well | Everything else -- most PyTorch/course workflows |

If you don't have a strong reason to use conda, use `venv` -- it's simpler to reason about and
easier for an AI agent (Claude Code, Codex) to manage on your behalf, since there's no separate
activation subsystem to get wrong.

## Step 0: Verify GPU Access

From a login-node terminal (or inside an interactive/batch job -- see
`srun_interactive_gpu_guide.ipynb` / `sbatch_job_submission_guide.ipynb`), run:

```bash
nvidia-smi
```

Example output on an Explorer H200 node:

```
+-----------------------------------------------------------------------------------+
| NVIDIA-SMI 550.xx    Driver Version: 550.xx    CUDA Version: 12.8                  |
|-------------------------------+----------------------+------------------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M.  |
|===============================+======================+======================|
|   0  NVIDIA H200            Off | 00000000:01:00.0 Off |                  0   |
| 32%   41C    P0    75W / 700W |     0MiB / 144564MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
```

If `nvidia-smi` isn't found, you're likely on a CPU-only login shell -- request a GPU via
`srun`/`sbatch` first.

## Step 1 (Option A -- venv, recommended): Create Your Environment

```bash
module load python/3.10
python -m venv ~/envs/ml_course_env
source ~/envs/ml_course_env/bin/activate
python -m pip install --upgrade pip
```

Reactivate it anytime (including from a fresh SSH session or an sbatch script) with:
```bash
source ~/envs/ml_course_env/bin/activate
```

## Step 1 (Option B -- conda): Create Your Environment

```bash
module load anaconda3
conda create -n ml_course_env python=3.10 -y
conda activate ml_course_env
```

## Step 2: Install PyTorch with GPU Support

Explorer's H200/A100 nodes run CUDA 12.8 drivers. Install a current PyTorch build against a
matching CUDA wheel (works whether you're in a `venv` or a conda env -- `pip install` is the same
either way):

```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

> The `cu121` wheels are forward-compatible with the CUDA 12.8 driver on Explorer -- you do not
> need an exact version match between the PyTorch CUDA build and the driver's reported CUDA
> version. See `H200-CU128.md` / `A100-CU128.md` in this repo for fuller per-GPU install recipes
> (TensorFlow, Hugging Face stack, Flash Attention, etc.).

This replaces the old Discovery-era pattern of `conda install pytorch==1.11.0 cudatoolkit=11.3` --
that CUDA/PyTorch combination predates Explorer and will not use the H200s correctly.

## Step 3: Verify GPU Availability with PyTorch

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name: ", torch.cuda.get_device_name(0))
    print("GPU memory (GB): ", torch.cuda.get_device_properties(0).total_memory / 1024**3)
else:
    print("No GPU available -- are you on a GPU node? (see srun_interactive_gpu_guide.ipynb)")

This will print whether a GPU is available and, if so, its name (e.g. `NVIDIA H200`) and
memory. On Explorer that should read ~141 GB for an H200, ~80 GB for an A100.

## Step 4: Save Your Environment for Reproducibility

**venv:**
```bash
pip freeze > requirements.txt
# recreate elsewhere with: pip install -r requirements.txt
```

**conda:**
```bash
conda env export > environment.yml
# recreate elsewhere with: conda env create -f environment.yml
```

Commit this file to your project repo (not the environment itself) so your `MANIFEST.md` can
point to exactly how to reproduce your setup -- see the agentic workflow notebook
(`agentic_cluster_workflow.ipynb`) and this repo's `SKILL.md` for how an AI agent can manage this
whole loop for you.

Now that you have a working environment and verified GPU access, continue to the PyTorch GPU
tutorial (`interactive_pytorch_gpu.ipynb` / `pytorch_gpu_training_part2.ipynb`) or jump straight
to the agentic workflow notebook to have an agent handle job submission and monitoring for you.